## Tokenizing with code

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model('gpt-4.1-mini')

tokens = encoding.encode('Hello my name is andrew')

In [ ]:
tokens

In [ ]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f'{token_id} = {token_text}')

### Check random range of numbers, what text they hold.

In [ ]:
for i in range(500, 502):
    token_text = encoding.decode([i])
    print(f'{i} = {token_text}')

In [ ]:
encoding.decode([326])

### Tokenizer - Gemini

In [ ]:
from google import genai
from google.genai import types

### Local count tokens

In [ ]:
from google.genai import local_tokenizer

tokenizer = local_tokenizer.LocalTokenizer(model_name='gemini-2.5-flash')
result = tokenizer.count_tokens("What is your name?")
result

### Local compute tokens

In [ ]:
from google.genai import local_tokenizer

tokenizer = local_tokenizer.LocalTokenizer(model_name='gemini-2.5-flash')
result = tokenizer.compute_tokens("What is your name?")
result

## Another Topic

### The Illusion of 'memory'

In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if not api_key:
    print('No API key found')
elif not api_key.startswith('AIz'):
    print('API key does not start with AIz')
else:
    print('API key found, and looks good so far')


### Calling the LLM

In [ ]:
from openai import OpenAI

base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

openai = OpenAI(api_key=api_key, base_url=base_url)


In [ ]:
messages = [
    {'role': 'system', 'content': 'You are an helpful assistant'},
    {'role': 'user', 'content': 'Hi my name is Andrew!'}
]

response = openai.chat.completions.create(model='gemini-2.5-flash-lite', messages=messages)
response.choices[0].message.content


### Asking a follow-up question

In [ ]:
messages = [
    {'role': 'system', 'content': 'You are an helpful assistant'},
    {'role': 'user', 'content': 'What is my name'}
]

response = openai.chat.completions.create(model='gemini-2.5-flash-lite', messages=messages)
response.choices[0].message.content


### Every call to an LLM is completely STATELESS.

In [ ]:
messages = [
    {'role': 'system', 'content': 'You are an helpful assistant'},
    {'role': 'user', 'content': 'Hi my name is Andrew!'},
    {'role': 'assistant', 'content': "Hi Andrew! It's nice to meet you. How can I help you today?"},
    {'role': 'user', 'content': 'What is my name'}
]


In [ ]:
response = openai.chat.completions.create(model='gemini-2.5-flash-lite', messages=messages)
response.choices[0].message.content

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

